# Healthcare Claims Fraud Detection

## Provider-Level Feature Engineering

The fraud label is available at the provider level. Therefore, beneficiary and claim information are aggregated into provider-level features to create the final machine learning dataset.

In [23]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Training Datasets

In [24]:
train = pd.read_csv("../data/raw/Train.csv")
beneficiary = pd.read_csv("../data/raw/Train_Beneficiarydata.csv")
inpatient = pd.read_csv("../data/raw/Train_Inpatientdata.csv")
outpatient = pd.read_csv("../data/raw/Train_Outpatientdata.csv")
print("Datasets Loaded Successfully")

Datasets Loaded Successfully


## Convert Date Columns

In [25]:
beneficiary["DOB"] = pd.to_datetime(beneficiary["DOB"])
beneficiary["DOD"] = pd.to_datetime(beneficiary["DOD"])
inpatient["AdmissionDt"] = pd.to_datetime(inpatient["AdmissionDt"])
inpatient["DischargeDt"] = pd.to_datetime(inpatient["DischargeDt"])
print("Date Conversion Completed")

Date Conversion Completed


## Feature Engineering - Patient Age

In [26]:
beneficiary["Age"] = (pd.Timestamp("2009-12-01").year - beneficiary["DOB"].dt.year)
beneficiary["Age"].describe()

count    138556.000000
mean         73.128663
std          12.724354
min          26.000000
25%          68.000000
50%          74.000000
75%          81.000000
max         100.000000
Name: Age, dtype: float64

## Feature Engineering - Total Chronic Conditions

In [27]:
chronic_cols = [
    "ChronicCond_Alzheimer",
    "ChronicCond_Heartfailure",
    "ChronicCond_KidneyDisease",
    "ChronicCond_Cancer",
    "ChronicCond_ObstrPulmonary",
    "ChronicCond_Depression",
    "ChronicCond_Diabetes",
    "ChronicCond_IschemicHeart",
    "ChronicCond_Osteoporasis",
    "ChronicCond_rheumatoidarthritis",
    "ChronicCond_stroke"
]
beneficiary["TotalChronicConditions"] = (beneficiary[chronic_cols].eq(1).sum(axis=1))
beneficiary["TotalChronicConditions"].describe()

count    138556.000000
mean          3.739131
std           2.346638
min           0.000000
25%           2.000000
50%           4.000000
75%           5.000000
max          11.000000
Name: TotalChronicConditions, dtype: float64

## Feature Engineering - Deceased Beneficiary Indicator

In [28]:
beneficiary["IsDeceased"] = (beneficiary["DOD"].notna().astype(int))
beneficiary["IsDeceased"].value_counts()

IsDeceased
0    137135
1      1421
Name: count, dtype: int64

## Feature Engineering - Length of Stay

In [29]:
inpatient["LengthOfStay"] = (inpatient["DischargeDt"] - inpatient["AdmissionDt"]).dt.days
inpatient["LengthOfStay"].describe()

count    40474.000000
mean         5.665168
std          5.638538
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         35.000000
Name: LengthOfStay, dtype: float64

## Feature Engineering - Diagnosis Count

In [30]:
diagnosis_cols = [
    "ClmDiagnosisCode_1",
    "ClmDiagnosisCode_2",
    "ClmDiagnosisCode_3",
    "ClmDiagnosisCode_4",
    "ClmDiagnosisCode_5",
    "ClmDiagnosisCode_6",
    "ClmDiagnosisCode_7",
    "ClmDiagnosisCode_8",
    "ClmDiagnosisCode_9",
    "ClmDiagnosisCode_10"
]
inpatient["DiagnosisCount"] = (inpatient[diagnosis_cols].notna().sum(axis=1))
inpatient["DiagnosisCount"].describe()

count    40474.000000
mean         8.087365
std          1.851830
min          1.000000
25%          8.000000
50%          9.000000
75%          9.000000
max         10.000000
Name: DiagnosisCount, dtype: float64

## Aggregate Inpatient Claims by Provider

In [31]:
inpatient_provider_features = (
    inpatient.groupby("Provider")
    .agg(
        InpatientClaimCount=("ClaimID", "count"),
        UniqueInpatientBeneficiaries=("BeneID", "nunique"),
        AvgInpatientReimbursement=("InscClaimAmtReimbursed", "mean"),
        TotalInpatientReimbursement=("InscClaimAmtReimbursed", "sum"),
        AvgLengthOfStay=("LengthOfStay", "mean"),
        MaxLengthOfStay=("LengthOfStay", "max"),
        AvgDiagnosisCount=("DiagnosisCount", "mean")
    )
    .reset_index()
)
inpatient_provider_features.head()

,Provider,InpatientClaimCount,UniqueInpatientBeneficiaries,AvgInpatientReimbursement,TotalInpatientReimbursement,AvgLengthOfStay,MaxLengthOfStay,AvgDiagnosisCount
0,PRV51001,5,5,19400.000000,97000,5.000000,14,7.200000
1,PRV51003,62,53,9241.935484,573000,5.161290,27,8.112903
2,PRV51007,3,3,6333.333333,19000,5.333333,7,7.333333
3,PRV51008,2,2,12500.000000,25000,4.000000,5,7.500000
4,PRV51011,1,1,5000.000000,5000,5.000000,5,8.000000


## Aggregate Outpatient Claims by Provider

In [32]:
outpatient_provider_features = (
    outpatient.groupby("Provider")
    .agg(
        OutpatientClaimCount=("ClaimID", "count"),
        UniqueOutpatientBeneficiaries=("BeneID", "nunique"),
        AvgOutpatientReimbursement=("InscClaimAmtReimbursed", "mean"),
        TotalOutpatientReimbursement=("InscClaimAmtReimbursed", "sum")
    )
    .reset_index()
)
outpatient_provider_features.head()

,Provider,OutpatientClaimCount,UniqueOutpatientBeneficiaries,AvgOutpatientReimbursement,TotalOutpatientReimbursement
0,PRV51001,20,19,382.000000,7640
1,PRV51003,70,66,466.714286,32670
2,PRV51004,149,138,350.134228,52170
3,PRV51005,1165,495,241.124464,280910
4,PRV51007,69,56,213.188406,14710


## Create Provider-Beneficiary Mapping

In [33]:
provider_beneficiary = pd.concat(
    [inpatient[["Provider", "BeneID"]], outpatient[["Provider", "BeneID"]]],
    ignore_index=True
).drop_duplicates()
provider_beneficiary.shape

(363300, 2)

## Merge Beneficiary Information

In [34]:
beneficiary_provider = provider_beneficiary.merge(beneficiary, on="BeneID", how="left")
beneficiary_provider.shape

(363300, 29)

## Aggregate Beneficiary Characteristics by Provider

In [35]:
beneficiary_provider_features = (
    beneficiary_provider.groupby("Provider")
    .agg(
        AvgPatientAge=("Age", "mean"),
        AvgChronicConditions=("TotalChronicConditions", "mean"),
        DeceasedRate=("IsDeceased", "mean"),
        AvgIPAnnualReimbursement=("IPAnnualReimbursementAmt", "mean"),
        AvgOPAnnualReimbursement=("OPAnnualReimbursementAmt", "mean"),
        UniqueBeneficiaries=("BeneID", "nunique")
    )
    .reset_index()
)
beneficiary_provider_features.head()

,Provider,AvgPatientAge,AvgChronicConditions,DeceasedRate,AvgIPAnnualReimbursement,AvgOPAnnualReimbursement,UniqueBeneficiaries
0,PRV51001,78.125000,5.541667,0.000000,18047.916667,2537.500000,24
1,PRV51003,68.957265,4.367521,0.008547,6814.017094,2490.598291,117
2,PRV51004,72.485507,4.318841,0.007246,4596.739130,2095.144928,138
3,PRV51005,69.987879,3.884848,0.006061,3717.232323,1798.808081,495
4,PRV51007,67.982759,3.879310,0.017241,3109.655172,1497.241379,58


## Build Final Provider Modeling Dataset

In [36]:
provider_df = (
    train
    .merge(inpatient_provider_features, on="Provider", how="left")
    .merge(outpatient_provider_features, on="Provider", how="left")
    .merge(beneficiary_provider_features, on="Provider", how="left")
)
provider_df.shape

(5410, 19)

## Assess Missing Values

In [37]:
provider_df.isnull().sum().sort_values(ascending=False)

InpatientClaimCount              3318
UniqueInpatientBeneficiaries     3318
AvgInpatientReimbursement        3318
TotalInpatientReimbursement      3318
AvgLengthOfStay                  3318
MaxLengthOfStay                  3318
AvgDiagnosisCount                3318
OutpatientClaimCount              398
UniqueOutpatientBeneficiaries     398
AvgOutpatientReimbursement        398
TotalOutpatientReimbursement      398
DeceasedRate                        0
AvgOPAnnualReimbursement            0
AvgIPAnnualReimbursement            0
Provider                            0
AvgChronicConditions                0
AvgPatientAge                       0
PotentialFraud                      0
UniqueBeneficiaries                 0
dtype: int64

## Handle Missing Values

In [38]:
provider_df.fillna(0, inplace=True)
provider_df.isnull().sum().sum()

np.int64(0)

## Encode Target Variable

In [39]:
provider_df["Fraud"] = (
    provider_df["PotentialFraud"]
    .map({
        "No": 0,
        "Yes": 1
    })
)
provider_df[["PotentialFraud", "Fraud"]].head()

,PotentialFraud,Fraud
0,No,0
1,Yes,1
2,No,0
3,Yes,1
4,No,0


## Final Dataset Overview

In [40]:
print("Shape:",provider_df.shape)
provider_df.info()

Shape: (5410, 20)
<class 'pandas.DataFrame'>
RangeIndex: 5410 entries, 0 to 5409
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Provider                       5410 non-null   str    
 1   PotentialFraud                 5410 non-null   str    
 2   InpatientClaimCount            5410 non-null   float64
 3   UniqueInpatientBeneficiaries   5410 non-null   float64
 4   AvgInpatientReimbursement      5410 non-null   float64
 5   TotalInpatientReimbursement    5410 non-null   float64
 6   AvgLengthOfStay                5410 non-null   float64
 7   MaxLengthOfStay                5410 non-null   float64
 8   AvgDiagnosisCount              5410 non-null   float64
 9   OutpatientClaimCount           5410 non-null   float64
 10  UniqueOutpatientBeneficiaries  5410 non-null   float64
 11  AvgOutpatientReimbursement     5410 non-null   float64
 12  TotalOutpatientReimbursement   5410 non-n

## Save Processed Dataset

In [41]:
import os
os.makedirs("../data/processed", exist_ok=True)
provider_df.to_csv("../data/processed/provider_features.csv", index=False)
print("Dataset Saved Successfully")

Dataset Saved Successfully


## Summary

Provider-level feature engineering completed successfully.

### Features Created

- Age
- Total Chronic Conditions
- IsDeceased
- Length Of Stay
- Diagnosis Count

### Provider-Level Aggregations

- Inpatient Claim Metrics
- Outpatient Claim Metrics
- Beneficiary Characteristics
- Reimbursement Metrics

### Output

provider_features.csv

This dataset will be used for provider-level exploratory analysis and machine learning modeling.